In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("AverageProcessingTime") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 09:09:15 WARN Utils: Your hostname, ast-ubuntu, resolves to a loopback address: 127.0.1.1; using 192.168.1.10 instead (on interface wlp2s0)
26/04/22 09:09:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 09:09:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
data_headers = ["Region", "Product_category", "Prod_id", "Revenue"]

In [4]:
data = [
    ("North", "Electronics", "E001", 1000),
    ("North", "Electronics", "E002", 1500),
    ("North", "Clothing", "C001", 500),
    ("South", "Electronics", "E003", 2000),
    ("South", "Clothing", "C002", 800),
    ("East", "Electronics", "E004", 1200),
    ("East", "Clothing", "C003", 600),      
    ("West", "Electronics", "E005", 1800),
    ("West", "Clothing", "C004", 700)
]   

In [5]:
input_df = spark.createDataFrame(data, schema=data_headers)

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [16]:
ws_region = Window.partitionBy("Region")
ws_category = Window.partitionBy("Product_category")


In [20]:
agg_df = input_df.withColumn("Region_max", F.max(F.col("Revenue")).over(ws_region) ) \
                .withColumn("Category_max", F.max(F.col("Revenue")).over(ws_category)  ) \
                .orderBy("Region","Product_category")

In [21]:
agg_df.show()

+------+----------------+-------+-------+----------+------------+
|Region|Product_category|Prod_id|Revenue|Region_max|Category_max|
+------+----------------+-------+-------+----------+------------+
|  East|        Clothing|   C003|    600|      1200|         800|
|  East|     Electronics|   E004|   1200|      1200|        2000|
| North|        Clothing|   C001|    500|      1500|         800|
| North|     Electronics|   E001|   1000|      1500|        2000|
| North|     Electronics|   E002|   1500|      1500|        2000|
| South|        Clothing|   C002|    800|      2000|         800|
| South|     Electronics|   E003|   2000|      2000|        2000|
|  West|        Clothing|   C004|    700|      1800|         800|
|  West|     Electronics|   E005|   1800|      1800|        2000|
+------+----------------+-------+-------+----------+------------+



In [24]:
agg_df2 = input_df.groupBy(F.col("Region")).agg(F.max(F.col("Revenue")).alias("Region_max"))
agg_df2.show()
agg_df3 = input_df.groupBy(F.col("Product_category")).agg(F.max(F.col("Revenue")).alias("Cat_max"))
agg_df3.show()


+------+----------+
|Region|Region_max|
+------+----------+
| North|      1500|
| South|      2000|
|  East|      1200|
|  West|      1800|
+------+----------+



+----------------+-------+
|Product_category|Cat_max|
+----------------+-------+
|     Electronics|   2000|
|        Clothing|    800|
+----------------+-------+



In [ ]:
spark.stop()